In [ ]:
import os
import random

import numpy as np
import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

# Quantization


In [66]:
def quantize(input_tensor, quantize_to_bits=8):
    @tf.custom_gradient
    def straight_through_estimator(input_tensor):
        max_int = (2 ** (quantize_to_bits - 1)) - 1
        max_float = tf.cast(max_int, tf.float32)
        max_val = tf.reduce_max(tf.abs(input_tensor))
        max_val = tf.maximum(max_val, 1e-8)

        scale = max_float / max_val

        quantized = tf.round(input_tensor * scale)
        quantized = tf.clip_by_value(quantized, -max_float, max_float)
        output = quantized / scale

        def grad(upstream, variables=None):
            if variables is not None:
                return upstream, [None] * len(variables)
            return upstream

        return output, grad

    return straight_through_estimator(input_tensor)

In [67]:
from keras.losses import mean_squared_error


def test_quantization():
    weights = tf.Variable([[1.02583, -0.238905], [-0.05612, -1.2983]], dtype=tf.float32)
    print("Original weights:\n", weights.numpy())

    target = tf.constant([[1.0, 1.0], [1.0, -1.0]], dtype=tf.float32)

    with tf.GradientTape() as tape:
        quantized_weights = quantize(weights, quantize_to_bits=4)
        loss = mean_squared_error(target, quantized_weights)
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, weights)

    print("\nQuantized weights:\n", quantized_weights.numpy())
    print("\nGradients:\n", gradients.numpy())


test_quantization()

Original weights:
 [[ 1.02583  -0.238905]
 [-0.05612  -1.2983  ]]

Quantized weights:
 [[ 1.1128286  -0.18547143]
 [-0.         -1.2983    ]]

Gradients:
 [[ 0.05641431 -0.5927357 ]
 [-0.5        -0.14915001]]


# Custom Quantized Dense Layer


In [68]:
from keras.layers import Layer


class QuantizedDense(Layer):
    def __init__(self, units, quantize_to_bits=8, **kwargs):
        super(QuantizedDense, self).__init__(**kwargs)
        self.units = units
        self.quantize_to_bits = quantize_to_bits

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_shape[-1], self.units),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.bias = self.add_weight(
            name="bias", shape=(self.units,), initializer="zeros", trainable=True
        )

    def call(self, inputs):
        quantized_kernel = quantize(self.kernel, quantize_to_bits=self.quantize_to_bits)
        output = tf.matmul(inputs, quantized_kernel)
        output += self.bias
        return output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.units)

    def get_config(self):
        config = super(QuantizedDense, self).get_config()
        config.update({"units": self.units, "quantize_to_bits": self.quantize_to_bits})
        return config

In [124]:
from keras import Input, Sequential

tf.random.set_seed(42)


def test_quantized_dense():
    model = Sequential([Input(shape=(4,)), QuantizedDense(units=3, quantize_to_bits=4)])

    test_input = tf.random.normal((2, 4))
    test_target = tf.random.normal((2, 3))

    with tf.GradientTape() as tape:
        predictions = model(test_input)
        loss = mean_squared_error(test_target, predictions)
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, model.trainable_variables)

    print("\nTest X-Y:\n", f"X: {test_input.numpy()}\n", f"Y: {test_target.numpy()}")

    print("\nModel shape:\n", predictions.shape)
    print("\nWeights:\n", model.layers[0].kernel.numpy())
    print("\nGradients", gradients[0].numpy())


test_quantized_dense()


Test X-Y:
 X: [[ 0.3274685 -0.8426258  0.3194337 -1.4075519]
 [-2.3880599 -1.0392479 -0.5573232  0.539707 ]]
 Y: [[ 0.08422458 -0.86090374  0.37812304]
 [-0.00519627 -0.49453196  0.6178192 ]]

Model shape:
 (2, 3)

Weights:
 [[-0.58286923 -0.5383417   0.20623076]
 [-0.15165418  0.35980535 -0.2821101 ]
 [-0.4641912  -0.49912745  0.6066592 ]
 [-0.47055316 -0.3550455  -0.7855339 ]]

Gradients [[-1.1354898  -1.1572453   1.3643099 ]
 [-0.59759516 -0.7413496   0.17866936]
 [-0.23948166 -0.21143359  0.42078632]
 [ 0.11659074 -0.06027183 -0.8701866 ]]
